In [5]:
import torch
import torch.nn.functional as F
from torch import nn

In [4]:
class RotatoryEmbedding(nn.Module):
    """Computes Rotary Positional Embeddings (RoPE) for sequence tokens."""

    def __init__(self, dim, max_sequence_length=2048):
        super().__init__()
        frequencies = 10000 * torch.arange(0, dim, 2).float() / dim
        inverse_frequency = 1 / frequencies
        t = torch.arange(max_sequence_length, dtype=torch.float32)
        freqs = torch.outer(t, inverse_frequency)
        self.register_buffer("cos", freqs.cos())
        self.register_buffer("sin", freqs.sin())

    def _rotate_half(self, x):
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat((-x2, x1), dim=-1)

    def forward(self, x: torch.Tensor, sequence_length=2048):
        # x shape: [Batch, Heads, Seq_Len, Head_Dim]
        B, H, dim = x.shape
        cos = self.cos[:sequence_length, None, :]  # [Seq_Len, 1, Head_Dim/2]
        sine = self.sin[:sequence_length, None, :]
        # Tile for matching dimensions
        cos = torch.cat([cos, cos], dim=-1).transpose(0, 1)  # [1,Seq_Len,Head_Dim]
        sine = torch.cat([sine, sine], dim=-1).transpose(0, 1)

        return x * cos + (self._rotate_half(x) * sine)